# 2. Descriptive Analysis & Data Preparation 

The original raw dataset contains over 1 million rows, which makes it challenging to process efficiently. To streamline the analysis, a random sampling method will be used to reduce the dataset size to 10% of its total, resulting in a final subset of 100,000 rows.

In [23]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', nrows=1048575)
df_sampled = df.sample(n=100000, random_state=42)
df_sampled.to_csv('sampled_data.csv', index=False)

The final dataset consists of 100,000 rows, prepared for preprocessing and statistical analysis. As the first step, the dataset is inspected for missing values and any necessary cleaning is applied to ensure data quality before analysis.

**Loading Sampled Dataset**

In [ ]:
# Load the sampled data
df = pd.read_csv('sampled_data.csv')

**Preliminary Inspection**

In [27]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   VendorID               100000 non-null  int64  
 1   tpep_pickup_datetime   100000 non-null  object 
 2   tpep_dropoff_datetime  100000 non-null  object 
 3   passenger_count        100000 non-null  int64  
 4   trip_distance          100000 non-null  float64
 5   RatecodeID             100000 non-null  int64  
 6   store_and_fwd_flag     100000 non-null  object 
 7   PULocationID           100000 non-null  int64  
 8   DOLocationID           100000 non-null  int64  
 9   payment_type           100000 non-null  int64  
 10  fare_amount            100000 non-null  float64
 11  extra                  100000 non-null  float64
 12  mta_tax                100000 non-null  float64
 13  tip_amount             100000 non-null  float64
 14  tolls_amount           100000 non-nul

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,1.680870,1.579980,3.137875,1.073690,162.133520,160.109320,1.328880,12.975659,1.025539,0.492498,2.079281,0.379273,0.297732,18.782503,2.252817
std,0.466142,1.186279,4.227822,0.855633,66.305034,71.008149,0.502375,13.554593,1.225608,0.074278,2.800640,1.711828,0.036228,16.087955,0.769476
min,1.000000,0.000000,-0.710000,1.000000,1.000000,1.000000,1.000000,-123.500000,-4.500000,-0.500000,-0.660000,-6.120000,-0.300000,-123.800000,-2.500000
25%,1.000000,1.000000,1.000000,1.000000,114.000000,107.000000,1.000000,6.000000,0.000000,0.500000,0.000000,0.000000,0.300000,10.800000,2.500000
50%,2.000000,1.000000,1.680000,1.000000,161.000000,162.000000,1.000000,9.000000,0.500000,0.500000,1.750000,0.000000,0.300000,14.000000,2.500000
75%,2.000000,2.000000,3.180000,1.000000,233.000000,233.000000,2.000000,14.000000,2.500000,0.500000,2.750000,0.000000,0.300000,19.560000,2.500000
max,2.000000,9.000000,259.220000,99.000000,265.000000,265.000000,4.000000,1238.000000,7.000000,3.300000,164.700000,96.420000,0.300000,1242.300000,2.750000


**Key Observations:**

* The dataset consists of 15 columns, including details like VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, trip_distance, and fare_amount.
* No missing or null values were identified in the dataset.
* The dataset contained data types such as integers, floats, and objects.
* There are no duplicate rows in the dataset. Each row represents a unique trip.
* There are some columns with negetive values which is likely due to errors.
* Some trips data contains high trip duration which seems to be outlier.


# 2.1 Data Preparation

**Correct Data type Errors**: 

Two critical fields i.e `tpep_pickup_datetime` and `tpep_dropoff_datetime` contains invalid data types which should be changed for time-based analysis(H4: Time of Day and Trip Duration/Fare).  

In [31]:
# Converting date columns to datetime objects
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

**Handle Negative Values:**

Some of the rows in the dataset contain negative values, which are illogical and cannot be true. I will drop these values to maintain data integrity.

In [35]:
# Select numeric columns
numeric_cols = df.select_dtypes(include=['number'])

# Check for negative values in numeric columns
negative_values = (numeric_cols < 0)

# Display summary of negative values per column
print("Summary of negative values in numeric columns:")
print(negative_values.sum())

# Remove rows with any negative values in numeric columns
df = df[~negative_values.any(axis=1)]

print(f"Original number of rows: {df.shape[0]}")
print(f"Number of rows after removing negatives: {df.shape[0]}")

Summary of negative values in numeric columns:
VendorID                 0
passenger_count          0
trip_distance            0
RatecodeID               0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
dtype: int64
Original number of rows: 99645
Number of rows after removing negatives: 99645


**Analysis of Long-Distance Trip**

This analysis is performing here to check the value contains in the trip distance is outlier or not.

In [37]:
long_distance_trips = df[df['trip_distance'] > 25]
print(long_distance_trips[['trip_distance']].head())

     trip_distance
293          50.00
294          27.29
382          27.30
557          25.71
597          28.38


In [39]:
# Filter for long trips where trip_distance > 200
long_trip = df[df['trip_distance'] > 200]

# Display detailed information about the long trips
print("Long trips with trip_distance > 200:")
print(long_trip[['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID',
                 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount',
                 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
                 'improvement_surcharge', 'total_amount', 'congestion_surcharge']])


Long trips with trip_distance > 200:
       VendorID  passenger_count  trip_distance  RatecodeID  PULocationID  \
49214         2                4         259.22           5           140   

       DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  \
49214           265             2        575.0    0.0      0.0         0.0   

       tolls_amount  improvement_surcharge  total_amount  congestion_surcharge  
49214           0.0                    0.3         577.8                   2.5  


A long-distance trip with a trip distance of 259.22 miles and a fare amount of $575 was recorded for 4 passengers under RatecodeID 5, suggesting a negotiated or special fare. Despite the unusual 50-hour duration, likely due to traffic or delays, the fare and associated congestion surcharge are consistent with long-distance travel, making this a valid data point for analysis.

**New Independent variable Creation:**

This steps makes our dataset more simpler which can be helpful for statistical analysis later.

**Trip Duration:**

Trip duration will be converted into minutes to study the impact of trip length on fare and revenue.

In [42]:
#Creating trip_duration feature
df['trip_duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60  # Duration in minutes

**Distance Category:**

Categorized trip_distance into bins (Very Short, Short, Medium, Long, Very Long) to simplify analysis and explore distance-related trends in fares and passenger behavior.

In [44]:
df['distance_category'] = pd.cut(pd.to_numeric(df['trip_distance'], errors='coerce'), bins=[0, 2, 5, 10, 20, np.inf], labels=['Very Short', 'Short', 'Medium', 'Long', 'Very Long'])
df['time_of_day'] = pd.cut(df['tpep_pickup_datetime'].dt.hour, bins=[0, 6, 18, 24], labels=['Night', 'Day', 'Evening'], include_lowest=True, duplicates='drop')

**Time of Day:** 

The time of day can significantly impact taxi fares and trip durations. This feature helps in analyzing these temporal effects.

In [46]:
# Create time_of_day feature (for H4)
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour
df['time_of_day'] = pd.cut(df['hour_of_day'], bins=[0, 6, 18, 24], labels=['Night', 'Day', 'Evening'], include_lowest=True, duplicates='drop')

**Rush Hour:** 

Rush hour can affect trip durations and fares due to traffic congestion. This feature helps in analyzing the impact of peak hours.

**Other reasons:**

* Analyzing the impact of traffic congestion on trip duration and fare amounts.
* Understanding passenger travel patterns and behavior during peak hours.
* Optimizing operational efficiency for taxi services.

In [48]:
# Create rush hour indicator
def is_rush_hour(hour):
    return 7 <= hour <= 9 or 16 <= hour <= 19
# Using 'hour_of_day' column to create rush_hour feature
df['rush_hour'] = df['hour_of_day'].apply(is_rush_hour)

**Data Saving**

In [50]:
df.to_csv('preprocessed_data.csv', index=False)

Now processed dataset is ready for the detailed descriptive analysis and statistical analysis.